#Transformar Dados de Sprints

- 1.Ler a tabela bronze sprints
- 2.Manter apenas as colunas necessárias para análise (Remover a coluna url)
- 3.Padronizar os nomes das colunas usando snake_case (constructorId → constructor_id, driverId → driver_id, raceName → race_name, positionText → finish_position_text)
- 4.Renomear colunas para torná-las mais significativas (date → race_date, grid → grid_position, laps → completed_laps, number → car_number, position → finish_position)
- 5.Filtrar linhas onde season, round, custructor_id ou driver_id estejam nulos (validação de chave de negócio)
- 6.Remover registros duplicados
- 7.Transformar os valores da coluna race_name para Title Case (Primeira Letra Maiúscula)
- 8.Escrever os dados transformados na tabela silver sprints

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/03.silver-helpers

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.sprints"
silver_table = f"{catalog_name}.{silver_schema}.sprints"

In [0]:
sprints_df = (
    spark.table(bronze_table).filter((F.col("batch_id")== v_batch_id))
    .drop("url")
    .withColumnsRenamed({
        "constructorId": "constructor_id",
        "driverId": "driver_id",
        "raceName": "race_name",
        "positionText": "finish_position_text",
        "date": "race_date",
        "grid": "grid_position",
        "laps": "completed_laps",
        "number": "car_number",
        "position": "finish_position"
    })
)

In [0]:
sprints_null_df = (
    sprints_df
    .filter(
        F.col("season").isNotNull() &
        F.col("round").isNotNull() &
        F.col("constructor_id").isNotNull() &
        F.col("driver_id").isNotNull ()
    )
    .dropDuplicates(["season", "round", "constructor_id", "driver_id"])
)


In [0]:
display(sprints_df.count() - sprints_null_df.count())

In [0]:
sprints_final_df = (
    sprints_null_df
    .withColumn("race_name", F.initcap(F.col("race_name")))
)

In [0]:
display(sprints_final_df)

In [0]:
write_to_silver(
    input_df=sprints_final_df,
    target_table=silver_table,
    merge_condition="t.season = s.season AND t.round = s.round AND t.constructor_id = s.constructor_id AND t.driver_id = s.driver_id",
    columns_to_update=[
       "race_date",
       "race_name",
       "grid_position",
       "completed_laps",
       "car_number",
       "points",
       "finish_position",
       "finish_position_text",
       "status",
       "ingestion_timestamp",
       "source_file",
       "batch_id" 
    ]
)

In [0]:
display(spark.table(silver_table))

race_date,race_name,round,season,constructor_id,driver_id,grid_position,completed_laps,car_number,points,finish_position,finish_position_text,status,ingestion_timestamp,source_file,batch_id,created_timestamp,updated_timestamp
2023-04-30,Azerbaijan Grand Prix,4,2023,red_bull,max_verstappen,3,17,1,6.0,3,null,Finished,2026-09-11T23:00:04.788Z,dbfs:/Volumes/formula1_incr/landing/arquivos/2025-01/sprints/sprints_2023.json,2025-01,2026-09-12T16:16:23.798Z,2026-09-12T16:16:23.798Z
2023-04-30,Azerbaijan Grand Prix,4,2023,williams,albon,7,17,23,0.0,9,null,Finished,2026-09-11T23:00:04.788Z,dbfs:/Volumes/formula1_incr/landing/arquivos/2025-01/sprints/sprints_2023.json,2025-01,2026-09-12T16:16:23.798Z,2026-09-12T16:16:23.798Z
2023-04-30,Azerbaijan Grand Prix,4,2023,alfa,zhou,14,17,24,0.0,12,null,Finished,2026-09-11T23:00:04.788Z,dbfs:/Volumes/formula1_incr/landing/arquivos/2025-01/sprints/sprints_2023.json,2025-01,2026-09-12T16:16:23.798Z,2026-09-12T16:16:23.798Z
2023-04-30,Azerbaijan Grand Prix,4,2023,alphatauri,tsunoda,16,2,22,0.0,19,null,Collision damage,2026-09-11T23:00:04.788Z,dbfs:/Volumes/formula1_incr/landing/arquivos/2025-01/sprints/sprints_2023.json,2025-01,2026-09-12T16:16:23.798Z,2026-09-12T16:16:23.798Z
2023-07-02,Austrian Grand Prix,9,2023,ferrari,leclerc,9,24,16,0.0,12,null,Finished,2026-09-11T23:00:04.788Z,dbfs:/Volumes/formula1_incr/landing/arquivos/2025-01/sprints/sprints_2023.json,2025-01,2026-09-12T16:16:23.798Z,2026-09-12T16:16:23.798Z
2023-07-30,Belgian Grand Prix,12,2023,aston_martin,stroll,14,11,18,0.0,11,null,Finished,2026-09-11T23:00:04.788Z,dbfs:/Volumes/formula1_incr/landing/arquivos/2025-01/sprints/sprints_2023.json,2025-01,2026-09-12T16:16:23.798Z,2026-09-12T16:16:23.798Z
2023-10-22,United States Grand Prix,18,2023,haas,hulkenberg,16,19,27,0.0,15,null,Finished,2026-09-11T23:00:04.788Z,dbfs:/Volumes/formula1_incr/landing/arquivos/2025-01/sprints/sprints_2023.json,2025-01,2026-09-12T16:16:23.798Z,2026-09-12T16:16:23.798Z
2023-11-05,São Paulo Grand Prix,20,2023,red_bull,max_verstappen,2,24,1,8.0,1,null,Finished,2026-09-11T23:00:04.788Z,dbfs:/Volumes/formula1_incr/landing/arquivos/2025-01/sprints/sprints_2023.json,2025-01,2026-09-12T16:16:23.798Z,2026-09-12T16:16:23.798Z
2024-05-05,Miami Grand Prix,6,2024,mclaren,piastri,6,19,81,3.0,6,null,Finished,2026-09-11T23:00:04.788Z,dbfs:/Volumes/formula1_incr/landing/arquivos/2025-01/sprints/sprints_2024.json,2025-01,2026-09-12T16:16:23.798Z,2026-09-12T16:16:23.798Z
2024-05-05,Miami Grand Prix,6,2024,haas,kevin_magnussen,14,19,20,0.0,18,null,Finished,2026-09-11T23:00:04.788Z,dbfs:/Volumes/formula1_incr/landing/arquivos/2025-01/sprints/sprints_2024.json,2025-01,2026-09-12T16:16:23.798Z,2026-09-12T16:16:23.798Z
